# 06 — FAISS Indexing Pipeline

Convert precomputed embeddings into searchable FAISS indexes.

**Inputs:**
- `data/processed/products_ml_ready.csv`
- `data/processed/image_embeddings.npy` — CLIP image embeddings, (4681, 512)
- `data/processed/image_embedding_index.csv`
- `data/processed/clip_text_embeddings.npy` — CLIP text embeddings, generated here if missing
- `data/processed/clip_text_embedding_index.csv`

**Outputs:**
- `data/processed/faiss/image_index.faiss`
- `data/processed/faiss/text_index.faiss`
- `data/processed/faiss/image_index_mapping.csv`
- `data/processed/faiss/text_index_mapping.csv`

Both indexes use **Inner Product** metric (equivalent to cosine similarity since embeddings are L2-normalized).
Both indexes share the same **512-dimensional** CLIP embedding space — essential for multimodal search later.

## 1. Imports

In [1]:
%pip install --break-system-packages faiss-cpu --quiet

import os
import numpy as np
import pandas as pd
import faiss
import torch
from pathlib import Path
from transformers import CLIPProcessor, CLIPTokenizer, CLIPModel
from tqdm.auto import tqdm
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"faiss version : {faiss.__version__}")
print(f"Device        : {DEVICE}")

Note: you may need to restart the kernel to use updated packages.


faiss version : 1.15.0
Device        : cpu


## 2. Paths

In [2]:
PROCESSED = Path("../data/processed")
FAISS_DIR = PROCESSED / "faiss"
FAISS_DIR.mkdir(parents=True, exist_ok=True)

# Input files
PRODUCTS_CSV        = PROCESSED / "products_ml_ready.csv"
IMAGE_EMB_NPY       = PROCESSED / "image_embeddings.npy"
IMAGE_IDX_CSV       = PROCESSED / "image_embedding_index.csv"
TEXT_EMB_NPY        = PROCESSED / "clip_text_embeddings.npy"
TEXT_IDX_CSV        = PROCESSED / "clip_text_embedding_index.csv"

# Output files
IMAGE_FAISS         = FAISS_DIR / "image_index.faiss"
TEXT_FAISS          = FAISS_DIR / "text_index.faiss"
IMAGE_MAPPING_CSV   = FAISS_DIR / "image_index_mapping.csv"
TEXT_MAPPING_CSV    = FAISS_DIR / "text_index_mapping.csv"

print("FAISS output directory:", FAISS_DIR.resolve())

FAISS output directory: D:\Ecommerce_search_engine\data\processed\faiss


## 3. Load Products and Image Embeddings

In [3]:
df = pd.read_csv(PRODUCTS_CSV)
image_embeddings = np.load(IMAGE_EMB_NPY)
image_idx_df = pd.read_csv(IMAGE_IDX_CSV)

print(f"Products loaded          : {len(df)}")
print(f"Image embedding shape    : {image_embeddings.shape}")
print(f"Image index rows         : {len(image_idx_df)}")

assert len(df) == 4681
assert image_embeddings.shape == (4681, 512), f"Expected (4681,512), got {image_embeddings.shape}"
assert len(image_idx_df) == 4681
print("\nImage embedding assertions passed.")

Products loaded          : 4681
Image embedding shape    : (4681, 512)
Image index rows         : 4681

Image embedding assertions passed.


## 4. Generate CLIP Text Embeddings (if not already saved)

The existing `text_embeddings.npy` uses SentenceTransformer (384-dim).  
For the FAISS indexes to share the same 512-dim CLIP space as the image embeddings, we need CLIP text embeddings.

We use the `combined_text_clean` column from `products_ml_ready.csv` and encode with the same CLIP model used for images: `openai/clip-vit-base-patch32`.

In [4]:
if TEXT_EMB_NPY.exists() and TEXT_IDX_CSV.exists():
    print("CLIP text embeddings already exist — loading from disk.")
    clip_text_embeddings = np.load(TEXT_EMB_NPY)
    text_idx_df = pd.read_csv(TEXT_IDX_CSV)
    print(f"Loaded shape: {clip_text_embeddings.shape}")
else:
    print("Generating CLIP text embeddings...")

    MODEL_NAME = "openai/clip-vit-base-patch32"
    clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
    clip_tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
    clip_model.eval()

    texts = df["combined_text_clean"].fillna("").tolist()
    BATCH_SIZE = 128
    all_text_embs = []
    text_index_rows = []
    current_idx = 0

    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="CLIP text encoding"):
        batch_texts = texts[start: start + BATCH_SIZE]
        batch_pids  = df["pid"].iloc[start: start + BATCH_SIZE].tolist()

        # CLIP tokenizer truncates at 77 tokens automatically
        inputs = clip_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            text_out = clip_model.text_model(**inputs)
            # pooler_output → text_projection gives the CLIP text embedding
            features = clip_model.text_projection(text_out.pooler_output)  # (B, 512)

        features = features.cpu().float().numpy()
        all_text_embs.append(features)

        for pid in batch_pids:
            text_index_rows.append({"embedding_index": current_idx, "pid": pid})
            current_idx += 1

    clip_text_embeddings = np.vstack(all_text_embs)  # (4681, 512)

    # L2 normalize
    norms = np.linalg.norm(clip_text_embeddings, axis=1, keepdims=True)
    clip_text_embeddings = clip_text_embeddings / np.clip(norms, 1e-10, None)

    # Save
    np.save(TEXT_EMB_NPY, clip_text_embeddings)
    text_idx_df = pd.DataFrame(text_index_rows)
    text_idx_df.to_csv(TEXT_IDX_CSV, index=False)

    print(f"CLIP text embeddings generated and saved.")
    print(f"Shape: {clip_text_embeddings.shape}")

    # Free model memory
    del clip_model, clip_tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

Generating CLIP text embeddings...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP text encoding:   0%|          | 0/37 [00:00<?, ?it/s]

CLIP text embeddings generated and saved.
Shape: (4681, 512)


## 5. Verify All Embeddings Before Indexing

In [5]:
def verify_embeddings(name, emb, idx_df, expected_shape):
    print(f"--- {name} ---")
    print(f"  Shape     : {emb.shape}")
    print(f"  Dtype     : {emb.dtype}")
    print(f"  Index rows: {len(idx_df)}")
    print(f"  NaN       : {np.isnan(emb).sum()}")
    print(f"  Inf       : {np.isinf(emb).sum()}")
    norms = np.linalg.norm(emb, axis=1)
    print(f"  Norms     : min={norms.min():.4f}  max={norms.max():.4f}  (1.0 = normalized)")
    dup_pids = idx_df["pid"].duplicated().sum()
    missing_pids = idx_df["pid"].isna().sum()
    print(f"  Dup PIDs  : {dup_pids}")
    print(f"  Miss PIDs : {missing_pids}")

    assert emb.shape == expected_shape, f"Shape mismatch: {emb.shape} != {expected_shape}"
    assert np.isnan(emb).sum() == 0, "NaN values found!"
    assert np.isinf(emb).sum() == 0, "Inf values found!"
    assert dup_pids == 0, "Duplicate PIDs found!"
    assert missing_pids == 0, "Missing PIDs found!"
    print(f"  ✓ All checks passed.\n")

# Ensure float32 — FAISS requires it
image_embeddings    = image_embeddings.astype(np.float32)
clip_text_embeddings = clip_text_embeddings.astype(np.float32)

verify_embeddings("Image Embeddings", image_embeddings,    image_idx_df, (4681, 512))
verify_embeddings("CLIP Text Embeddings", clip_text_embeddings, text_idx_df, (4681, 512))

--- Image Embeddings ---
  Shape     : (4681, 512)
  Dtype     : float32
  Index rows: 4681
  NaN       : 0
  Inf       : 0
  Norms     : min=1.0000  max=1.0000  (1.0 = normalized)
  Dup PIDs  : 0
  Miss PIDs : 0
  ✓ All checks passed.

--- CLIP Text Embeddings ---
  Shape     : (4681, 512)
  Dtype     : float32
  Index rows: 4681
  NaN       : 0
  Inf       : 0
  Norms     : min=1.0000  max=1.0000  (1.0 = normalized)
  Dup PIDs  : 0
  Miss PIDs : 0
  ✓ All checks passed.



## 6. Build Image FAISS Index

Using `IndexFlatIP` — exact inner product search.  
Since embeddings are L2-normalized, inner product = cosine similarity.

In [6]:
DIM = 512  # CLIP embedding dimension

# IndexFlatIP = exact inner product search (cosine sim for normalized vectors)
image_index = faiss.IndexFlatIP(DIM)
image_index.add(image_embeddings)

print(f"Image FAISS index built")
print(f"  Dimension : {image_index.d}")
print(f"  Vectors   : {image_index.ntotal}")
assert image_index.ntotal == 4681

Image FAISS index built
  Dimension : 512
  Vectors   : 4681


## 7. Build Text FAISS Index

In [7]:
text_index = faiss.IndexFlatIP(DIM)
text_index.add(clip_text_embeddings)

print(f"Text FAISS index built")
print(f"  Dimension : {text_index.d}")
print(f"  Vectors   : {text_index.ntotal}")
assert text_index.ntotal == 4681

Text FAISS index built
  Dimension : 512
  Vectors   : 4681


## 8. Save Indexes and Mapping Files

In [8]:
# Save FAISS indexes
faiss.write_index(image_index, str(IMAGE_FAISS))
faiss.write_index(text_index,  str(TEXT_FAISS))

# Save mapping files: faiss_index → pid
# image_idx_df already has embedding_index + pid columns
image_mapping = image_idx_df[["embedding_index", "pid"]].copy()
image_mapping.columns = ["faiss_index", "pid"]

text_mapping = text_idx_df[["embedding_index", "pid"]].copy()
text_mapping.columns = ["faiss_index", "pid"]

image_mapping.to_csv(IMAGE_MAPPING_CSV, index=False)
text_mapping.to_csv(TEXT_MAPPING_CSV,   index=False)

img_sz  = os.path.getsize(IMAGE_FAISS) / (1024*1024)
text_sz = os.path.getsize(TEXT_FAISS)  / (1024*1024)

print(f"Saved image_index.faiss       → {IMAGE_FAISS}  ({img_sz:.1f} MB)")
print(f"Saved text_index.faiss        → {TEXT_FAISS}   ({text_sz:.1f} MB)")
print(f"Saved image_index_mapping.csv → {IMAGE_MAPPING_CSV}")
print(f"Saved text_index_mapping.csv  → {TEXT_MAPPING_CSV}")

Saved image_index.faiss       → ..\data\processed\faiss\image_index.faiss  (9.1 MB)
Saved text_index.faiss        → ..\data\processed\faiss\text_index.faiss   (9.1 MB)
Saved image_index_mapping.csv → ..\data\processed\faiss\image_index_mapping.csv
Saved text_index_mapping.csv  → ..\data\processed\faiss\text_index_mapping.csv


## 9. Structural Validation

In [9]:
def validate_index(name, path, expected_dim, expected_ntotal):
    assert path.exists(), f"{name}: file not found at {path}"
    idx = faiss.read_index(str(path))
    assert idx.d      == expected_dim,   f"{name}: dim {idx.d} != {expected_dim}"
    assert idx.ntotal == expected_ntotal, f"{name}: ntotal {idx.ntotal} != {expected_ntotal}"
    print(f"  {name}: dim={idx.d}, ntotal={idx.ntotal} ✓")

def validate_mapping(name, path, expected_rows):
    assert path.exists(), f"{name}: file not found"
    m = pd.read_csv(path)
    assert len(m) == expected_rows,        f"{name}: rows {len(m)} != {expected_rows}"
    assert m["pid"].isna().sum()  == 0,   f"{name}: missing PIDs"
    assert m["pid"].duplicated().sum() == 0, f"{name}: duplicate PIDs"
    print(f"  {name}: rows={len(m)}, no missing PIDs, no duplicates ✓")

print("=== Structural Validation ===")
validate_index("Image Index", IMAGE_FAISS, 512, 4681)
validate_index("Text Index",  TEXT_FAISS,  512, 4681)
validate_mapping("Image Mapping", IMAGE_MAPPING_CSV, 4681)
validate_mapping("Text Mapping",  TEXT_MAPPING_CSV,  4681)
print("\nAll structural checks passed.")

=== Structural Validation ===
  Image Index: dim=512, ntotal=4681 ✓
  Text Index: dim=512, ntotal=4681 ✓
  Image Mapping: rows=4681, no missing PIDs, no duplicates ✓
  Text Mapping: rows=4681, no missing PIDs, no duplicates ✓

All structural checks passed.


## 10. Reload Test

Delete in-memory indexes, reload from disk, confirm they work.

In [10]:
# Delete in-memory indexes
del image_index, text_index

# Reload from disk
image_index_reloaded = faiss.read_index(str(IMAGE_FAISS))
text_index_reloaded  = faiss.read_index(str(TEXT_FAISS))

print("Reloaded from disk:")
print(f"  image_index.ntotal : {image_index_reloaded.ntotal}")
print(f"  text_index.ntotal  : {text_index_reloaded.ntotal}")

assert image_index_reloaded.ntotal == 4681
assert text_index_reloaded.ntotal  == 4681
print("\nReload test passed.")

Reloaded from disk:
  image_index.ntotal : 4681
  text_index.ntotal  : 4681

Reload test passed.


## Summary

In [11]:
print("=" * 44)
print("FAISS INDEXING COMPLETE")
print("=" * 44)
print(f"Products indexed : 4681")
print()
print("Text Index:")
print(f"  Dimension  : {text_index_reloaded.d}")
print(f"  Vectors    : {text_index_reloaded.ntotal}")
print(f"  Metric     : Inner Product (cosine sim)")
print(f"  Saved      : {TEXT_FAISS}")
print(f"  Reloaded   : OK")
print()
print("Image Index:")
print(f"  Dimension  : {image_index_reloaded.d}")
print(f"  Vectors    : {image_index_reloaded.ntotal}")
print(f"  Metric     : Inner Product (cosine sim)")
print(f"  Saved      : {IMAGE_FAISS}")
print(f"  Reloaded   : OK")
print()
print("Mappings:")
print(f"  Text mapping  : 4681 rows  → {TEXT_MAPPING_CSV}")
print(f"  Image mapping : 4681 rows  → {IMAGE_MAPPING_CSV}")
print()
print("CLIP text embeddings:")
print(f"  Saved : {TEXT_EMB_NPY}")
print(f"  Index : {TEXT_IDX_CSV}")
print()
print(f"Output directory : {FAISS_DIR.resolve()}")
print("=" * 44)

FAISS INDEXING COMPLETE
Products indexed : 4681

Text Index:
  Dimension  : 512
  Vectors    : 4681
  Metric     : Inner Product (cosine sim)
  Saved      : ..\data\processed\faiss\text_index.faiss
  Reloaded   : OK

Image Index:
  Dimension  : 512
  Vectors    : 4681
  Metric     : Inner Product (cosine sim)
  Saved      : ..\data\processed\faiss\image_index.faiss
  Reloaded   : OK

Mappings:
  Text mapping  : 4681 rows  → ..\data\processed\faiss\text_index_mapping.csv
  Image mapping : 4681 rows  → ..\data\processed\faiss\image_index_mapping.csv

CLIP text embeddings:
  Saved : ..\data\processed\clip_text_embeddings.npy
  Index : ..\data\processed\clip_text_embedding_index.csv

Output directory : D:\Ecommerce_search_engine\data\processed\faiss
